# Sentiment Analysis Project
## Data Loading

In [1]:
import bz2

file_path = "C:/Users/shrey/Downloads/archive/train.ft.txt.bz2"

with bz2.open(file_path, 'rt', encoding='utf-8') as f:
    lines = f.readlines()

len(lines)

3600000

## Data Conversion (FastText → DataFrame)

In [4]:
import pandas as pd

data = []

for line in lines:
    parts = line.split(' ', 1)
    label = parts[0]
    text = parts[1] if len(parts) > 1 else ""

    sentiment = 1 if label == "__label__2" else 0

    data.append([text.strip(), sentiment])

df = pd.DataFrame(data, columns=["review", "sentiment"])

df.head()

,review,sentiment
0,Stuning even for the non-gamer: This sound tra...,1
1,The best soundtrack ever to anything.: I'm rea...,1
2,Amazing!: This soundtrack is my favorite music...,1
3,Excellent Soundtrack: I truly like this soundt...,1
4,"Remember, Pull Your Jaw Off The Floor After He...",1


## Reduce Dataset Size

In [7]:
df = df.head(50000)

## Data Cleaning

In [10]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

df['clean_text'] = df['review'].apply(clean_text)

## Train-Test Split

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['sentiment'], test_size=0.2, random_state=42
)

## Feature Extraction (TF-IDF)

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

## Model Training (Logistic Regression)

In [21]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train_vec, y_train)

LogisticRegression()

## Model Evaluation

In [24]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test_vec)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.89      0.88      0.88      4955
           1       0.88      0.90      0.89      5045

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



## Save Model

In [43]:
import pickle

pickle.dump(model, open("models/sentiment_model.pkl", "wb"))
pickle.dump(vectorizer, open("models/vectorizer.pkl", "wb"))

## Confusion Matrix

In [32]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, y_pred))

[[4348  607]
 [ 524 4521]]
